# 04 — Model Evaluation
Load the saved model, run evaluation on the test set, and generate all performance plots.

In [ ]:
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))
import pandas as pd
import joblib
import config
from src.features import add_behavioral_features
from src.evaluate import full_evaluation, evaluate_model, plot_confusion_matrix, plot_roc_curve, plot_feature_importance
%matplotlib inline

## 1. Load Test Data & Model

In [ ]:
X_test  = pd.read_csv(config.X_TEST_PROCESSED)
y_test  = pd.read_csv(config.Y_TEST_PATH).squeeze()
model   = joblib.load(config.MODEL_PATH)
print('Test shape:', X_test.shape)
print('Model loaded:', type(model).__name__)

## 2. Apply Feature Engineering

In [ ]:
X_test = add_behavioral_features(X_test)
print('Features after engineering:', X_test.shape[1])

## 3. Full Evaluation (Metrics + Plots)

In [ ]:
metrics = full_evaluation(model, X_test, y_test)
print('\nFinal Metrics:', metrics)

## 4. Threshold Analysis

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

y_prob = model.predict_proba(X_test)[:, 1]
thresholds = np.arange(0.1, 0.9, 0.05)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_t, zero_division=0))
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))

plt.figure(figsize=(9, 5))
plt.plot(thresholds, precisions, label='Precision', marker='o')
plt.plot(thresholds, recalls,    label='Recall',    marker='s')
plt.plot(thresholds, f1s,        label='F1-Score',  marker='^')
plt.xlabel('Decision Threshold')
plt.ylabel('Score')
plt.title('Precision / Recall / F1 vs Threshold')
plt.legend()
plt.tight_layout()
os.makedirs(config.FIGURES_DIR, exist_ok=True)
plt.savefig(os.path.join(config.FIGURES_DIR, 'threshold_analysis.png'), dpi=150)
plt.show()

## 5. Summary

In [ ]:
print('=== UPI Fraud Detection — Final Results ===')
for k, v in metrics.items():
    print(f'  {k:<12}: {v}')